In [ ]:
# ============================
# 🚂 TRAIN — обучаем EI и сохраняем артефакты
# ============================

import os, json, re, warnings
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple


# ---------- Конфиг ----------
START_DATE = "2023-01-01"
DRIVE_URL  = "https://drive.google.com/file/d/1TUEKqTs8cSnWcf1_nk8W1fJXqKIB3xJ-/view?usp=sharing"  # CSV

# EI/обучение
H_FWD_DAYS = 7
EPS = 1.0
MIN_ABS_DLOGP = 0.05
MAD_K = 3.5
EI_CLIP = (0.05, 5.0)
NMF_COMPONENTS = 3
SAMPLE_MAX = 1_200_000
CAT_MODELS_MIN_ROWS = 30_000

# Локальная медиана: фиксированное окно ±K по рангу
RANK_WINDOW_K = 10

# «Шоки»
UPDATE_HALF_LIFE  = 5
UPDATE_TAIL_DAYS  = 14
FEAT_HALF_LIFE    = 7
FEAT_TAIL_DAYS    = 21
FEAT_ORG_SPIKE_RATIO = 2.0
FEAT_PAID_GROW_MAX   = 0.2
FEAT_RANK_IMPROVE    = 30.0

# Папка для артефактов
ART_DIR = "ei_artifacts"

# ---------- Утилиты ----------
def _drive_file_id(url: str) -> str:
    m = re.search(r"/d/([^/]+)/", url or "")
    if not m:
        raise ValueError("Ожидаю ссылку вида .../file/d/<ID>/view")
    return m.group(1)

def fetch_from_drive(url: str, out_path: str = "googleplay_merged.csv") -> str:
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path
    import gdown
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    file_id = _drive_file_id(url)
    uc = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(uc, out_path, quiet=False)
    if not (os.path.exists(out_path) and os.path.getsize(out_path) > 0):
        raise IOError("Не удалось скачать CSV с Google Drive")
    return out_path

def safe_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

def mad_mask(x: np.ndarray, k: float = MAD_K) -> np.ndarray:
    x = x.astype(float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    z = (x - med) / (1.4826 * mad)
    return np.abs(z) <= k

# ---------- Загрузка данных ----------
def load_raw_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    rename_map = {
        "Date":"date","Organic Search":"organic_search","Organic Browse":"organic_browse",
        "Paid Ads":"paid_ads","Paid Search":"paid_search","Web Browser":"web_browser",
        "App ID":"app_id","Revenue ($)":"revenue","RPD ($)":"rpd","ARPDAU ($)":"arpdau",
        "DAU":"dau","Rank":"rank","Updated":"updated","Category":"category",
        "WasRanked":"was_ranked","Price":"price"
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df = df.rename(columns={k: v})

    df["date"] = safe_dt(df.get("date"))
    if df["date"].notna().any():
        df = df[df["date"] >= pd.Timestamp(START_DATE)].copy()

    # Абсолютные трафики
    df["organic_traffic_total"] = df.get("organic_search",0).fillna(0) + df.get("organic_browse",0).fillna(0)
    df["paid_traffic"] = df.get("paid_ads",0).fillna(0) + df.get("paid_search",0).fillna(0)

    for c in ["arpdau","revenue","dau","paid_traffic","web_browser","rpd","rank"]:
        df[c] = pd.to_numeric(df.get(c,0), errors="coerce").fillna(0.0)

    def to_flag(s: pd.Series) -> pd.Series:
        return (s.astype(str).str.strip().str.lower()
                .map({"1":1,"0":0,"true":1,"false":0,"yes":1,"no":0,"y":1,"n":0})
                .fillna(0).astype(int))
    for f in ["updated","was_ranked"]:
        df[f] = to_flag(df[f]) if f in df.columns else 0

    if "category" not in df.columns: df["category"] = "Unknown"
    if "price" in df.columns: df = df.drop(columns=["price"])

    df = df[(df["revenue"] >= 0) & (df["arpdau"] >= 0)]
    return df

# ---------- Признаки ----------
def add_logs(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for c in ["organic_traffic_total","paid_traffic","dau","revenue","rank"]:
        # лог безопасный: log(max(x, EPS))
        d[f"log_{c}"] = np.log(np.clip(d[c].astype(float), a_min=EPS, a_max=None))
    d["sRank"] = -d["log_rank"]  # меньше — лучше
    return d

def _rolling_median(s: pd.Series, w=7):
    return s.rolling(w, min_periods=1).median()

def detect_suspect_featuring(d: pd.DataFrame) -> pd.Series:
    g = d.groupby("app_id", group_keys=False)
    org_ma = g["organic_traffic_total"].apply(_rolling_median)
    paid_ma = g["paid_traffic"].apply(_rolling_median)
    org_ratio = (d["organic_traffic_total"] / (org_ma + 1e-6)).fillna(0.0)
    paid_growth = ((d["paid_traffic"] - paid_ma).abs() / (paid_ma + 1e-6)).fillna(0.0)

    rank_improve = pd.Series(0.0, index=d.index)
    for L in [1,2,3]:
        prev = g["rank"].shift(L)
        imp = (prev - d["rank"])
        rank_improve = np.maximum(rank_improve, imp.fillna(0.0))

    spike = (org_ratio > FEAT_ORG_SPIKE_RATIO) & (paid_growth <= FEAT_PAID_GROW_MAX)
    good_rank = (rank_improve >= FEAT_RANK_IMPROVE)
    suspect = (spike | good_rank).astype(int)

    sus_g = suspect.groupby(d["app_id"])
    return ((suspect + sus_g.shift(1).fillna(0) + sus_g.shift(-1).fillna(0)) > 0).astype(int)

def _days_since_last_event(flag: pd.Series, dates: pd.Series, groups: pd.Series) -> pd.Series:
    dates = safe_dt(dates)
    last = np.where(flag.values.astype(int)==1, dates.values, pd.NaT)
    s = pd.Series(last, index=flag.index).groupby(groups).ffill()
    s = safe_dt(s)
    return (dates - s).dt.days

def _decay_kernel(days_since: pd.Series, half_life: float, tail_days: int) -> pd.Series:
    ds = days_since.copy()
    k = pd.Series(0.0, index=ds.index)
    m = (ds.notna()) & (ds >= 0) & (ds <= tail_days)
    k.loc[m] = np.power(2.0, -(ds.loc[m] / max(half_life, 1e-6)))
    return k

def add_event_shocks(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["date"] = safe_dt(d["date"])
    d = d.sort_values(["app_id","date"]).copy()

    upd_flag = d.get("updated",0).fillna(0).astype(int)
    ds_upd = _days_since_last_event(upd_flag, d["date"], d["app_id"])
    d["is_update_day"] = (upd_flag==1).astype(int)
    d["days_since_update"] = ds_upd.fillna(-1).astype(int).clip(-1, UPDATE_TAIL_DAYS)
    d["k_update"] = _decay_kernel(ds_upd, UPDATE_HALF_LIFE, UPDATE_TAIL_DAYS)

    sus = detect_suspect_featuring(d)
    ds_feat = _days_since_last_event(sus, d["date"], d["app_id"])
    d["is_feature_day"] = sus.astype(int)
    d["days_since_feature"] = ds_feat.fillna(-1).astype(int).clip(-1, FEAT_TAIL_DAYS)
    d["k_feature"] = _decay_kernel(ds_feat, FEAT_HALF_LIFE, FEAT_TAIL_DAYS)

    d["is_event_window_any"] = (
        (d["k_update"] > 0) | (d["k_feature"] > 0) |
        (d["is_update_day"]==1) | (d["is_feature_day"]==1)
    ).astype(int)
    return d

def build_ei_targets(df: pd.DataFrame, h=H_FWD_DAYS, min_abs_dlogp=MIN_ABS_DLOGP) -> pd.DataFrame:
    d = df.copy()
    d["date"] = safe_dt(d["date"])
    d = d.sort_values(["app_id","date"])
    d["dlogP"] = d.groupby("app_id")["log_paid_traffic"].diff()
    d["logO_fwd"] = d.groupby("app_id")["log_organic_traffic_total"].shift(-h)
    d["dlogO_fwd"] = d["logO_fwd"] - d["log_organic_traffic_total"]
    d["eta"] = d["dlogO_fwd"] / d["dlogP"]
    # фильтры
    d = d[(d["dlogP"].abs() >= min_abs_dlogp) & np.isfinite(d["eta"])]
    d = d.loc[mad_mask(d["eta"].to_numpy(), k=MAD_K)].copy()
    return d

# === Локальная медиана по фиксированному окну ±K ===
def normalize_by_category_week_local(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["date"] = safe_dt(d["date"])
    iso = d["date"].dt.isocalendar()
    d["iso_year"], d["iso_week"] = iso.year.astype(int), iso.week.astype(int)

    base = d[d["is_event_window_any"] == 0].copy()
    d["eta_med_local"] = np.nan

    from bisect import bisect_left, bisect_right
    for (cat, y, w), g_all in d.groupby(["category","iso_year","iso_week"]):
        g_base = base[(base["category"]==cat)&(base["iso_year"]==y)&(base["iso_week"]==w)]
        if len(g_base) < 5:
            continue
        g_base = g_base.sort_values("rank")
        ranks = g_base["rank"].to_numpy()
        etas  = g_base["eta"].to_numpy()

        def local_med(r):
            L = bisect_left(ranks, r - RANK_WINDOW_K)
            R = bisect_right(ranks, r + RANK_WINDOW_K)
            if R - L < 3:
                return np.nan
            return np.nanmedian(etas[L:R])

        g_all_sorted = g_all.sort_values("rank")
        meds = [local_med(r) for r in g_all_sorted["rank"].to_numpy()]
        d.loc[g_all_sorted.index, "eta_med_local"] = meds

    # строго финитная локальная медиана
    d = d[np.isfinite(d["eta_med_local"])].copy()

    # EI как отношение к локальной медиане (предохраняемся от нулей и ультра‑малых)
    denom = d["eta_med_local"].replace(0, np.nan)
    denom = np.where(np.isfinite(denom), denom, np.nan)
    d["EI"] = (d["eta"] / denom).astype(float)
    d["EI"] = pd.Series(d["EI"]).replace([np.inf, -np.inf], np.nan)
    d = d[d["EI"].notna()].copy()
    d["EI"] = d["EI"].clip(*EI_CLIP)

    # recency фича
    last_date = d["date"].max() or pd.Timestamp("today").normalize()
    d["age_days"] = (last_date - d["date"]).dt.days.astype(float)

    return d

# ---------- Латенты ----------
def fit_nmf_latents(df: pd.DataFrame, components: int = NMF_COMPONENTS):
    from sklearn.decomposition import NMF
    feats = ["log_paid_traffic","log_organic_traffic_total","log_dau","log_revenue","sRank"]
    X = df[feats].fillna(0.0).to_numpy()
    shift = -min(0.0, float(np.nanmin(X)))
    Xp = np.maximum(X + shift, 0.0)
    nmf = NMF(n_components=components, init="nndsvda", random_state=42, max_iter=400)
    Z = nmf.fit_transform(Xp)
    for i in range(components):
        df[f"z{i+1}"] = Z[:, i]
    info = {"nmf_shift": shift, "components": components}
    return df, nmf, info

def label_latents_by_corr(df_lat: pd.DataFrame) -> dict:
    anchors = {"Видимость/Ранг":"sRank", "Монетизация":"log_revenue", "Удержание/Вовлечённость":"log_dau"}
    labels = {}
    for zi in [c for c in df_lat.columns if c.startswith("z")]:
        best_name, best_abs = None, -1
        for name, col in anchors.items():
            c = np.corrcoef(df_lat[zi].fillna(0), df_lat[col].fillna(0))[0,1]
            if np.abs(c) > best_abs:
                best_abs, best_name = np.abs(c), name
        labels[zi] = best_name or "Качество ядра"
    return labels

def build_feature_matrix(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    feat_cols = [
        "log_paid_traffic","log_organic_traffic_total","log_dau","log_revenue","sRank",
        "dlogP","age_days",
        "is_update_day","days_since_update","k_update",
        "is_feature_day","days_since_feature","k_feature",
    ] + [c for c in df.columns if c.startswith("z")]
    for c in feat_cols:
        if c not in df.columns: df[c] = 0.0
    # подчистим inf
    X = df[feat_cols].replace([np.inf, -np.inf], np.nan)
    return X, feat_cols

# ---------- Обучение и сохранение ----------
def _make_clean_xy(df_lat: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """Единая чистка X/y: убираем NaN/Inf только из y; X оставляем (CatBoost сам справится с NaN)."""
    X, feat_cols = build_feature_matrix(df_lat)
    y = pd.to_numeric(df_lat["EI"], errors="coerce")
    y = y.replace([np.inf, -np.inf], np.nan)
    mask_ok = y.notna() & np.isfinite(y)
    Xc, yc = X.loc[mask_ok], y.loc[mask_ok].astype(float)
    # каппинг по объёму (равномерная подвыборка по времени)
    if len(Xc) > SAMPLE_MAX:
        # предпочтительно последние наблюдения (свежее)
        Xc = Xc.tail(SAMPLE_MAX)
        yc = yc.tail(SAMPLE_MAX)
    if len(Xc) < 1000:
        raise RuntimeError(f"Мало валидных строк после чистки таргета: {len(Xc)}")
    return Xc, yc

def train_and_save():
    warnings.filterwarnings("ignore")
    csv_path = fetch_from_drive(DRIVE_URL, out_path="googleplay_merged.csv")
    raw = load_raw_csv(csv_path)

    d = raw.copy()
    d["date"] = safe_dt(d["date"])
    d = d[d["date"].notna()].copy()

    d = add_logs(d)
    d = add_event_shocks(d)
    d = build_ei_targets(d)

    if d.empty or d["eta"].notna().sum() < 1000:
        raise RuntimeError("Мало сигналов dlogP/eta — ослабь MIN_ABS_DLOGP или проверь CSV.")

    d = normalize_by_category_week_local(d)

    d_lat, nmf, nmf_info = fit_nmf_latents(d.copy(), components=NMF_COMPONENTS)
    latent_labels = label_latents_by_corr(d_lat)

    # === ЕДИНАЯ чистка и формирование X/y для общего датасета ===
    X_all, y_all = _make_clean_xy(d_lat)

    from catboost import CatBoostRegressor, Pool

    def fit_q(alpha, X, y):
        model = CatBoostRegressor(
            loss_function=f"Quantile:alpha={alpha}",
            depth=8, learning_rate=0.06, iterations=1200,
            l2_leaf_reg=3.0, random_seed=42, verbose=False,
            allow_writing_files=False
        )
        model.fit(Pool(X, y))
        return model

    models_all = {
        "p25": fit_q(0.25, X_all, y_all),
        "p50": fit_q(0.50, X_all, y_all),
        "p75": fit_q(0.75, X_all, y_all)
    }

    # === По категориям (только крупные), с той же чисткой ===
    models_by_cat = {}
    sizes = d_lat["category"].value_counts()
    big_cats = sizes[sizes >= CAT_MODELS_MIN_ROWS].index.tolist()
    for cat in big_cats:
        dd = d_lat[d_lat["category"] == cat].copy()
        Xc, yc = _make_clean_xy(dd)
        models_by_cat[cat] = {
            "p25": fit_q(0.25, Xc, yc),
            "p50": fit_q(0.50, Xc, yc),
            "p75": fit_q(0.75, Xc, yc)
        }

    # === Сохранение ===
    os.makedirs(ART_DIR, exist_ok=True)
    import joblib
    joblib.dump(models_all, os.path.join(ART_DIR, "models_all.joblib"))
    joblib.dump(models_by_cat, os.path.join(ART_DIR, "models_by_cat.joblib"))
    joblib.dump(nmf, os.path.join(ART_DIR, "nmf.joblib"))

    meta = {
        "nmf_shift": nmf_info["nmf_shift"],
        "feat_cols": list(X_all.columns),
        "latent_labels": latent_labels,
        "rows_used": int(len(X_all)),
        "big_cats": big_cats,
        "config": {
            "H_FWD_DAYS": H_FWD_DAYS, "MIN_ABS_DLOGP": MIN_ABS_DLOGP,
            "RANK_WINDOW_K": RANK_WINDOW_K,
            "CAT_MODELS_MIN_ROWS": CAT_MODELS_MIN_ROWS,
            "EI_CLIP": EI_CLIP, "SAMPLE_MAX": SAMPLE_MAX
        }
    }
    with open(os.path.join(ART_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved artifacts to: {os.path.abspath(ART_DIR)}")
    print("Загрузи на Google Drive (anyone with link) файлы:")
    print(" - models_all.joblib")
    print(" - models_by_cat.joblib")
    print(" - nmf.joblib")
    print(" - meta.json")

# Запуск
train_and_save()
